# FinNexus — Exploratory Data Analysis

**Pipeline:** Data Cleaning → Feature Engineering → EDA → Cross-Asset Analysis

**Asset Categories:** Crypto · Indian Stocks · Commodities · ETFs · Futures & Options

---
**Contents**
1. Setup & Imports
2. Load Cleaned Data
3. Data Quality Overview
4. Single-Asset Deep Dive (Crypto: BTC)
5. Category-Level EDA — Crypto
6. Category-Level EDA — Commodities
7. Category-Level EDA — ETFs
8. Category-Level EDA — Indian Stocks
9. Category-Level EDA — Futures & Options
10. Cross-Asset Correlation Matrix
11. Performance & Risk Rankings
12. Feature Engineering
13. Feature Inspection & Target Variable
14. Key Insights Summary

## 1. Setup & Imports

In [ ]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats as scipy_stats

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (14, 5)
plt.style.use('seaborn-v0_8-darkgrid')

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT         = Path('..').resolve()
CLEANED_ROOT = ROOT / 'Data' / 'Cleaned'
FEATURES_ROOT= ROOT / 'Data' / 'Features'
REPORTS_ROOT = ROOT / 'Reports'

sys.path.insert(0, str(ROOT))

print('ROOT:', ROOT)
print('Cleaned categories:', [d.name for d in CLEANED_ROOT.iterdir() if d.is_dir()])

## 2. Load Cleaned Data
Load all cleaned CSVs from `Data/Cleaned/` into category dictionaries.

In [ ]:
def load_category(category: str) -> dict:
    """Load all *_cleaned.csv files for a category into {symbol: df}."""
    cat_dir = CLEANED_ROOT / category
    assets = {}
    for path in sorted(cat_dir.glob('*_cleaned.csv')):
        symbol = path.stem.replace('_cleaned', '')
        df = pd.read_csv(path)
        df['Date'] = pd.to_datetime(df['Date'])
        assets[symbol] = df
    print(f'  {category}: {len(assets)} assets loaded')
    return assets

crypto      = load_category('Crypto')
commodities = load_category('Commodities')
etfs        = load_category('ETFs')
stocks      = load_category('Stocks')
futures     = load_category('Futures')

## 3. Data Quality Overview

In [ ]:
def load_quality_report(category: str) -> pd.DataFrame:
    path = REPORTS_ROOT / f'Data_Quality_{category}.json'
    with open(path) as f:
        data = json.load(f)
    return pd.DataFrame(data)

quality_frames = {}
for cat in ['Crypto', 'Commodities', 'ETFs', 'Futures', 'Stocks']:
    qdf = load_quality_report(cat)
    quality_frames[cat] = qdf

# Summary table per category
summary_rows = []
for cat, qdf in quality_frames.items():
    scores = qdf['quality_score'].dropna()
    summary_rows.append({
        'Category': cat,
        'Assets': len(qdf),
        'Avg Quality': scores.mean().round(1),
        'Min Quality': scores.min().round(1),
        'Max Quality': scores.max().round(1),
        'Avg Missing %': qdf['missing_pct'].mean().round(2),
        'Total Outlier Rows': qdf['outlier_rows'].sum(),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Category')
display(summary_df)

In [ ]:
# Quality score distribution — bar chart per category
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
colors = ['#2980b9','#27ae60','#e67e22','#8e44ad','#e74c3c']
for ax, (cat, qdf), color in zip(axes, quality_frames.items(), colors):
    scores = qdf['quality_score'].dropna()
    ax.hist(scores, bins=15, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(scores.mean(), color='black', linewidth=1.5, linestyle='--', label=f'mean={scores.mean():.1f}')
    ax.set_title(cat, fontweight='bold')
    ax.set_xlabel('Quality Score')
    ax.legend(fontsize=8)
axes[0].set_ylabel('Count')
fig.suptitle('Data Quality Score Distribution by Category', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Single-Asset Deep Dive — BTC
Full price, returns, volume, and volatility analysis for Bitcoin.

In [ ]:
df = crypto['BTC'].copy()
dates = df['Date']
close = df['Close']
print(f'BTC: {len(df)} rows | {dates.min().date()} → {dates.max().date()}')
print(f'Price range: ${close.min():,.0f}  →  ${close.max():,.0f}')
display(df[['Date','Open','High','Low','Close','Volume']].tail(5))

In [ ]:
# ── Price Analysis ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('BTC — Price Analysis', fontsize=14, fontweight='bold')

# 1. Linear price
ax = axes[0, 0]
ax.plot(dates, close, linewidth=1, color='#2980b9')
ax.set_title('Price History (Linear)')
ax.set_ylabel('USD')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.tick_params(axis='x', rotation=30)

# 2. Log scale
ax = axes[0, 1]
ax.semilogy(dates, close, linewidth=1, color='#8e44ad')
ax.set_title('Price History (Log Scale)')
ax.set_ylabel('USD (log)')
ax.tick_params(axis='x', rotation=30)

# 3. Moving averages
ax = axes[1, 0]
ax.plot(dates, close, linewidth=0.8, label='Close', color='#2980b9', alpha=0.8)
for w, c in [(20,'#f39c12'),(50,'#27ae60'),(200,'#e74c3c')]:
    ax.plot(dates, close.rolling(w).mean(), linewidth=1, label=f'SMA{w}', alpha=0.85)
ax.set_title('Price + Moving Averages')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=30)

# 4. Drawdown
ax = axes[1, 1]
cum = (1 + close.pct_change().fillna(0)).cumprod()
dd  = (cum - cum.expanding().max()) / cum.expanding().max() * 100
ax.fill_between(dates, dd, 0, alpha=0.5, color='#e74c3c')
ax.plot(dates, dd, linewidth=0.7, color='#c0392b')
ax.set_title('Drawdown (%)')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ── Returns Analysis ────────────────────────────────────────────────────────
daily_ret = close.pct_change().dropna()
log_ret   = np.log(close / close.shift(1)).dropna()

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('BTC — Returns Analysis', fontsize=14, fontweight='bold')

# 1. Histogram
ax = axes[0, 0]
ax.hist(daily_ret * 100, bins=60, color='#27ae60', edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=1)
ax.axvline(daily_ret.mean()*100, color='red', linewidth=1.5, linestyle='--', label=f'mean={daily_ret.mean()*100:.2f}%')
ax.set_title('Daily Returns Distribution (%)')
ax.set_xlabel('Daily Return (%)')
ax.legend(fontsize=8)

# 2. Q-Q plot
ax = axes[0, 1]
osm, osr = scipy_stats.probplot(daily_ret.dropna(), dist='norm')
ax.scatter(osm[0], osm[1], s=4, alpha=0.5, color='#2980b9')
ax.plot([osm[0][0], osm[0][-1]],
        [osr[1]+osr[0]*osm[0][0], osr[1]+osr[0]*osm[0][-1]],
        color='red', linewidth=1.5)
ax.set_title('Q-Q Plot (Normality Check)')
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')

# 3. Rolling 30d return
ax = axes[1, 0]
r30 = close.pct_change(30) * 100
ax.plot(dates, r30, linewidth=0.8, color='#e67e22')
ax.axhline(0, color='black', linewidth=0.8)
ax.fill_between(dates, r30, 0, where=(r30>=0), alpha=0.3, color='#2ecc71')
ax.fill_between(dates, r30, 0, where=(r30<0),  alpha=0.3, color='#e74c3c')
ax.set_title('Rolling 30-Day Return (%)')
ax.tick_params(axis='x', rotation=30)

# 4. Rolling volatility
ax = axes[1, 1]
for w, col in [(7,'#3498db'),(30,'#e67e22'),(90,'#9b59b6')]:
    vol = log_ret.rolling(w).std() * np.sqrt(252) * 100
    ax.plot(dates[log_ret.index], vol, linewidth=0.8, label=f'Vol {w}d', color=col)
ax.set_title('Rolling Volatility (Annualised %)')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print(f'Skewness : {daily_ret.skew():.4f}')
print(f'Kurtosis : {daily_ret.kurt():.4f}  (excess over normal=3)')

In [ ]:
# ── Volume Analysis ─────────────────────────────────────────────────────────
vol = df['Volume'].replace(0, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('BTC — Volume Analysis', fontsize=13, fontweight='bold')

ax = axes[0]
ax.bar(dates, vol, width=1, color='#2980b9', alpha=0.6)
ax.plot(dates, vol.rolling(30).mean(), color='orange', linewidth=1.2, label='30d MA')
ax.set_title('Daily Volume')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=30)

ax = axes[1]
ax2 = ax.twinx()
ax.bar(dates, vol, width=1, color='#3498db', alpha=0.4)
ax2.plot(dates, close, linewidth=0.9, color='#e74c3c')
ax.set_title('Volume vs Price')
ax.set_ylabel('Volume', color='#3498db')
ax2.set_ylabel('Price', color='#e74c3c')
ax.tick_params(axis='x', rotation=30)

ax = axes[2]
vol_ratio = vol.rolling(7).mean() / vol.rolling(30).mean()
ax.plot(dates, vol_ratio, linewidth=0.8, color='#27ae60')
ax.axhline(1.0, color='black', linewidth=0.8, linestyle='--')
ax.axhline(2.0, color='red',   linewidth=0.8, linestyle='--', label='2x spike')
ax.set_title('Volume Ratio (7d / 30d avg)')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ── Risk Metrics ─────────────────────────────────────────────────────────────
from Backend.data.eda import compute_asset_stats

btc_stats = compute_asset_stats(df.set_index('Date') if df.index.name != 'Date' else df, 'BTC')
# compute_asset_stats expects Date as column, pass as-is
btc_stats = compute_asset_stats(df, 'BTC')

metrics_df = pd.DataFrame([{
    'Metric': k.replace('_', ' ').title(),
    'Value':  v
} for k, v in btc_stats.items() if k not in ('asset', 'start_date', 'end_date')])
display(metrics_df.set_index('Metric'))

## 5. Category-Level EDA — Crypto
Compare all 10 crypto assets: price history, returns distribution, and performance.

In [ ]:
from Backend.data.eda import compute_asset_stats

def category_stats(assets: dict) -> pd.DataFrame:
    rows = [compute_asset_stats(df, name) for name, df in assets.items()]
    return pd.DataFrame(rows).set_index('asset')

crypto_stats = category_stats(crypto)
display(crypto_stats[[
    'n_rows', 'price_last', 'total_return_pct', 'annual_return_pct',
    'ann_volatility', 'sharpe_ratio', 'max_drawdown_pct', 'var_95_pct'
]].sort_values('sharpe_ratio', ascending=False).round(3))

In [ ]:
# Normalised price index (all start at 100) for comparison
fig, ax = plt.subplots(figsize=(15, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(crypto)))

for (name, df), color in zip(crypto.items(), colors):
    close = df.set_index('Date')['Close'].dropna()
    normalised = close / close.iloc[0] * 100
    ax.plot(normalised.index, normalised, linewidth=1.2, label=name, color=color)

ax.axhline(100, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_title('Crypto — Normalised Price (Base=100 at Start)', fontsize=13, fontweight='bold')
ax.set_ylabel('Indexed Price')
ax.legend(fontsize=8, ncol=2)
ax.set_yscale('log')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Returns distribution comparison — boxplot
returns_data = {}
for name, df in crypto.items():
    returns_data[name] = df['Close'].pct_change().dropna() * 100

fig, ax = plt.subplots(figsize=(14, 5))
ax.boxplot(returns_data.values(), labels=returns_data.keys(),
           patch_artist=True, notch=True,
           boxprops=dict(facecolor='#3498db', alpha=0.6),
           medianprops=dict(color='red', linewidth=2),
           flierprops=dict(marker='.', markersize=3, alpha=0.4))
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Crypto — Daily Returns Distribution (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Daily Return (%)')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Volatility comparison bar chart
vol_data = {}
for name, df in crypto.items():
    log_r = np.log(df['Close'] / df['Close'].shift(1)).dropna()
    vol_data[name] = log_r.std() * np.sqrt(252) * 100

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(vol_data.keys(), vol_data.values(),
              color=['#2ecc71' if v < 80 else '#e74c3c' for v in vol_data.values()],
              edgecolor='white')
for bar, val in zip(bars, vol_data.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}%', ha='center', va='bottom', fontsize=9)
ax.set_title('Crypto — Annualised Volatility (%)', fontsize=13, fontweight='bold')
ax.set_ylabel('Ann. Volatility (%)')
plt.tight_layout()
plt.show()

## 6. Category-Level EDA — Commodities

In [ ]:
comm_stats = category_stats(commodities)
display(comm_stats[[
    'n_rows', 'total_return_pct', 'annual_return_pct',
    'ann_volatility', 'sharpe_ratio', 'max_drawdown_pct'
]].sort_values('sharpe_ratio', ascending=False).round(3))

In [ ]:
# Normalised price (exclude Lead — sparse data)
fig, ax = plt.subplots(figsize=(15, 6))
colors = plt.cm.Set2(np.linspace(0, 1, len(commodities)))

for (name, df), color in zip(commodities.items(), colors):
    if name == 'Lead':   # low quality — skip
        continue
    close = df.set_index('Date')['Close'].dropna()
    norm  = close / close.iloc[0] * 100
    ax.plot(norm.index, norm, linewidth=1.1, label=name, color=color)

ax.axhline(100, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_title('Commodities — Normalised Price (Base=100)', fontsize=13, fontweight='bold')
ax.set_ylabel('Indexed Price')
ax.legend(fontsize=8, ncol=3)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Gold deep-dive — rolling volatility + Bollinger bands
gold = commodities['Gold'].copy()
close = gold['Close']
dates = gold['Date']
mid   = close.rolling(20).mean()
std   = close.rolling(20).std()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Gold — Bollinger Bands & Volatility', fontweight='bold')

ax = axes[0]
ax.plot(dates, close, linewidth=0.8, label='Close', color='#f1c40f')
ax.plot(dates, mid, linewidth=0.9, label='SMA20', color='orange')
ax.fill_between(dates, mid-2*std, mid+2*std, alpha=0.2, color='orange', label='BB ±2σ')
ax.set_title('Bollinger Bands (20d, ±2σ)')
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=30)

ax = axes[1]
log_r = np.log(close / close.shift(1))
vol30 = log_r.rolling(30).std() * np.sqrt(252) * 100
ax.plot(dates, vol30, linewidth=0.9, color='#e74c3c')
ax.fill_between(dates, vol30, alpha=0.3, color='#e74c3c')
ax.set_title('Gold Annualised Volatility (30d Rolling) %')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 7. Category-Level EDA — ETFs

In [ ]:
etf_stats = category_stats(etfs)
display(etf_stats[[
    'n_rows', 'total_return_pct', 'annual_return_pct',
    'ann_volatility', 'sharpe_ratio', 'max_drawdown_pct'
]].sort_values('sharpe_ratio', ascending=False).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
colors = plt.cm.tab20(np.linspace(0, 1, len(etfs)))

for (name, df), color in zip(etfs.items(), colors):
    close = df.set_index('Date')['Close'].dropna()
    norm  = close / close.iloc[0] * 100
    ax.plot(norm.index, norm, linewidth=1.1, label=name, color=color)

ax.axhline(100, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_title('ETFs — Normalised Price (Base=100)', fontsize=13, fontweight='bold')
ax.set_ylabel('Indexed Price')
ax.legend(fontsize=8, ncol=3)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Risk-Return scatter: ETFs
fig, ax = plt.subplots(figsize=(10, 7))
for name, row in etf_stats.iterrows():
    x = row.get('ann_volatility', np.nan)
    y = row.get('annual_return_pct', np.nan)
    if np.isnan(x) or np.isnan(y):
        continue
    ax.scatter(x * 100, y, s=100, zorder=5)
    ax.annotate(name, (x*100, y), textcoords='offset points',
                xytext=(6, 3), fontsize=8)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('Annual Return (%)')
ax.set_title('ETF Risk-Return Profile', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Category-Level EDA — Indian Stocks
All 294 Nifty50 / Midcap / Next50 / Smallcap stocks.

In [ ]:
stock_stats = category_stats(stocks)
print(f'Total stocks: {len(stock_stats)}')

# Top 15 by Sharpe
print('\nTop 15 stocks by Sharpe Ratio:')
display(stock_stats[[
    'total_return_pct','annual_return_pct','ann_volatility','sharpe_ratio','max_drawdown_pct'
]].dropna(subset=['sharpe_ratio']).nlargest(15, 'sharpe_ratio').round(3))

In [ ]:
# Nifty 50 — index stocks category comparison
n50_symbols = [k for k in stocks if k.startswith('N50_')]
nmid_symbols = [k for k in stocks if k.startswith('NMidcap_')]
nnext_symbols = [k for k in stocks if k.startswith('NNext_')]
nsmall_symbols = [k for k in stocks if k.startswith('NSmallcap_')]

def group_sharpe(symbols):
    s = stock_stats.loc[[s for s in symbols if s in stock_stats.index], 'sharpe_ratio'].dropna()
    return s

fig, ax = plt.subplots(figsize=(12, 5))
groups  = {'N50': group_sharpe(n50_symbols),
           'NMidcap': group_sharpe(nmid_symbols),
           'NNext': group_sharpe(nnext_symbols),
           'NSmallcap': group_sharpe(nsmall_symbols)}

ax.boxplot(groups.values(), labels=groups.keys(),
           patch_artist=True, notch=True,
           boxprops=dict(facecolor='#3498db', alpha=0.6),
           medianprops=dict(color='red', linewidth=2))
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Sharpe Ratio Distribution by Index Category', fontsize=13, fontweight='bold')
ax.set_ylabel('Sharpe Ratio')
plt.tight_layout()
plt.show()

In [ ]:
# Stocks — Risk-Return scatter coloured by index group
group_map = {}
for sym in n50_symbols:    group_map[sym] = 'N50'
for sym in nmid_symbols:   group_map[sym] = 'NMidcap'
for sym in nnext_symbols:  group_map[sym] = 'NNext'
for sym in nsmall_symbols: group_map[sym] = 'NSmallcap'

palette = {'N50':'#2980b9','NMidcap':'#27ae60','NNext':'#e67e22','NSmallcap':'#9b59b6'}
fig, ax = plt.subplots(figsize=(12, 7))

for sym, row in stock_stats.iterrows():
    x = row.get('ann_volatility', np.nan)
    y = row.get('annual_return_pct', np.nan)
    if np.isnan(x) or np.isnan(y):
        continue
    grp = group_map.get(sym, 'Other')
    ax.scatter(x*100, y, s=20, alpha=0.6, color=palette.get(grp, 'grey'))

# Legend patches
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=c, label=g) for g,c in palette.items()], fontsize=9)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('Annual Return (%)')
ax.set_title('Stocks Risk-Return Profile (All 294 Stocks)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Category-Level EDA — Futures & Options

In [ ]:
fut_stats = category_stats(futures)
display(fut_stats[[
    'n_rows','total_return_pct','annual_return_pct','ann_volatility','sharpe_ratio','max_drawdown_pct'
]].round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
colors = plt.cm.Set1(np.linspace(0, 1, len(futures)))

for (name, df), color in zip(futures.items(), colors):
    close = df.set_index('Date')['Close'].dropna()
    norm  = close / close.iloc[0] * 100
    ax.plot(norm.index, norm, linewidth=1.1, label=name, color=color)

ax.axhline(100, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_title('Futures & Options — Normalised Price (Base=100)', fontsize=13, fontweight='bold')
ax.set_ylabel('Indexed Price')
ax.legend(fontsize=7, ncol=2)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 10. Cross-Asset Correlation Matrix
Pairwise correlation of daily returns across ALL asset categories.

In [ ]:
# Build a combined returns DataFrame — sample representative assets to keep it readable
representative = {
    # Crypto
    **{k: v for k, v in crypto.items()},
    # Commodities (exclude Lead)
    **{k: v for k, v in commodities.items() if k != 'Lead'},
    # ETFs
    **{k: v for k, v in etfs.items()},
    # Futures — index-level only
    'NIFTY_Futures': futures.get('NIFTY_50_Futures'),
    'BANKNIFTY_Futures': futures.get('BANK_NIFTY_Futures'),
    # Nifty 50 stocks — first 15 for readability
    **{k: v for k, v in list(stocks.items()) if k.startswith('N50_')}[:15]
}
representative = {k: v for k, v in representative.items() if v is not None}

# Compute daily return series
ret_series = {}
for name, df in representative.items():
    s = df.set_index('Date')['Close'].pct_change().rename(name)
    ret_series[name] = s

ret_df = pd.DataFrame(ret_series)
corr   = ret_df.corr()

print(f'Correlation matrix: {corr.shape[0]} x {corr.shape[1]} assets')

In [ ]:
fig, ax = plt.subplots(figsize=(max(12, len(corr)*0.5), max(10, len(corr)*0.45)))
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticklabels(corr.index, fontsize=7)

for i in range(len(corr)):
    for j in range(len(corr.columns)):
        val = corr.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=5,
                color='black' if abs(val) < 0.7 else 'white')

ax.set_title('Cross-Asset Correlation Matrix (Daily Returns)', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Key cross-category correlations
print('=== BTC vs other assets ===')
if 'BTC' in corr.columns:
    btc_corr = corr['BTC'].drop('BTC').sort_values(ascending=False)
    print(btc_corr.round(3).to_string())

print('\n=== Gold vs other assets ===')
if 'Gold' in corr.columns:
    gold_corr = corr['Gold'].drop('Gold').sort_values(ascending=False)
    print(gold_corr.round(3).to_string())

## 11. Performance & Risk Rankings
All assets ranked by Sharpe Ratio and other risk-adjusted metrics.

In [ ]:
# Combine all stats
all_stats = pd.concat([
    crypto_stats.assign(category='Crypto'),
    comm_stats.assign(category='Commodity'),
    etf_stats.assign(category='ETF'),
    fut_stats.assign(category='Futures'),
    stock_stats.assign(category='Stock'),
])

print(f'Total assets ranked: {len(all_stats)}')
print('\nTop 20 by Sharpe Ratio:')
display(all_stats[['category','sharpe_ratio','annual_return_pct','ann_volatility','max_drawdown_pct']]
        .dropna(subset=['sharpe_ratio'])
        .nlargest(20, 'sharpe_ratio')
        .round(3))

In [ ]:
# Top 25 — horizontal bar chart (Sharpe)
top25 = all_stats.dropna(subset=['sharpe_ratio']).nlargest(25, 'sharpe_ratio')

cat_colors = {'Crypto':'#2980b9','Commodity':'#f1c40f','ETF':'#27ae60',
              'Futures':'#e74c3c','Stock':'#8e44ad'}

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(top25.index, top25['sharpe_ratio'],
               color=[cat_colors.get(c,'grey') for c in top25['category']],
               edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Sharpe Ratio')
ax.set_title('Top 25 Assets by Sharpe Ratio', fontsize=13, fontweight='bold')
ax.invert_yaxis()

from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=c, label=g) for g,c in cat_colors.items()], fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Risk-Return scatter — all categories
fig, ax = plt.subplots(figsize=(14, 8))

for cat, subset in all_stats.groupby('category'):
    x = subset['ann_volatility'].dropna() * 100
    y = subset['annual_return_pct'].dropna()
    common = x.index.intersection(y.index)
    ax.scatter(x[common], y[common], label=cat,
               color=cat_colors.get(cat,'grey'), s=20, alpha=0.65)

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('Annual Return (%)')
ax.set_title('Risk-Return Profile — All Assets', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 12. Feature Engineering
Demonstrates the full feature pipeline on a single asset (BTC), then loads pre-built feature CSVs.

In [ ]:
from Backend.data.features import (
    create_technical_features,
    create_all_targets,
    create_context_features,
    prepare_features,
)

# Build features for BTC from cleaned data
btc_clean = crypto['BTC'].copy()
btc_feat  = create_technical_features(btc_clean)
btc_feat  = create_all_targets(btc_feat)
btc_feat  = prepare_features(btc_feat, target_col='target_7d')

print(f'BTC features: {len(btc_feat)} rows x {len(btc_feat.columns)} columns')
print('\nFeature groups:')
base = {'Date','Open','High','Low','Close','Volume','is_outlier','data_quality'}
feat_cols = [c for c in btc_feat.columns if c not in base]

groups = {
    'Returns':     [c for c in feat_cols if 'return' in c or 'momentum' in c or 'log_' in c],
    'Moving Avg':  [c for c in feat_cols if 'sma' in c or 'ema' in c or 'dist_from' in c or 'golden' in c or 'above_' in c],
    'Oscillators': [c for c in feat_cols if any(x in c for x in ['rsi','macd','stoch','adx','di_'])],
    'Bollinger':   [c for c in feat_cols if 'bb_' in c],
    'Volume':      [c for c in feat_cols if any(x in c for x in ['volume','obv','vwap','vpt'])],
    'Volatility':  [c for c in feat_cols if 'vol' in c or 'atr' in c or 'natr' in c or 'parkinson' in c],
    'Price Level': [c for c in feat_cols if any(x in c for x in ['price_to','day_range','gap'])],
    'Targets':     [c for c in feat_cols if 'target' in c],
}
for grp, cols in groups.items():
    print(f'  {grp:12s}: {len(cols)} — {cols}')

In [ ]:
# Load pre-computed feature CSV (faster for large datasets)
def load_features(category: str, symbol: str) -> pd.DataFrame:
    path = FEATURES_ROOT / category / f'{symbol}_features.csv'
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'])
    return df

btc_feat_csv  = load_features('Crypto',      'BTC')
gold_feat_csv = load_features('Commodities', 'Gold')
spy_feat_csv  = load_features('ETFs',        'SPY')
hdfc_feat_csv = load_features('Stocks',      'N50_HDFCBANK')

for label, df in [('BTC','btc_feat_csv'),('Gold','gold_feat_csv'),
                  ('SPY','spy_feat_csv'),('HDFC Bank','hdfc_feat_csv')]:
    d = eval(df)
    print(f'{label:12s}: {len(d)} rows x {len(d.columns)} cols')

## 13. Feature Inspection & Target Variable

In [ ]:
# Target variable balance check
fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
horizons  = [3, 5, 7, 10, 14]
colors    = ['#3498db','#27ae60','#e67e22','#9b59b6','#e74c3c']

for ax, h, color in zip(axes, horizons, colors):
    col = f'target_{h}d'
    if col not in btc_feat.columns:
        continue
    counts = btc_feat[col].value_counts().sort_index()
    ax.bar(['Down (0)','Up (1)'], counts.values, color=[color, '#2ecc71'], edgecolor='white')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 2, str(v), ha='center', fontsize=9)
    ax.set_title(f'Target {h}d', fontweight='bold')
    ax.set_xlabel(f'Balance: {counts.get(1,0)/len(btc_feat[col].dropna())*100:.1f}% Up')

axes[0].set_ylabel('Count')
fig.suptitle('BTC — Target Variable Balance by Horizon', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Technical indicator visualisation — RSI, MACD, Bollinger (last 200 rows)
df  = btc_feat.tail(200).copy()
dt  = df['Date']

fig, axes = plt.subplots(4, 1, figsize=(15, 14), sharex=True)
fig.suptitle('BTC — Technical Indicators (Last 200 Days)', fontsize=13, fontweight='bold')

# 1. Price + Bollinger
ax = axes[0]
ax.plot(dt, df['Close'], linewidth=1, color='#2980b9', label='Close')
if 'bb_upper' in df.columns:
    ax.fill_between(dt, df['bb_lower'], df['bb_upper'], alpha=0.2, color='orange', label='BB')
    ax.plot(dt, df['bb_middle'], linewidth=0.8, color='orange', linestyle='--')
ax.legend(fontsize=8)
ax.set_ylabel('Price')

# 2. RSI
ax = axes[1]
if 'rsi_14' in df.columns:
    ax.plot(dt, df['rsi_14'], linewidth=1, color='#9b59b6')
    ax.axhline(70, color='red',   linewidth=0.8, linestyle='--', label='Overbought 70')
    ax.axhline(30, color='green', linewidth=0.8, linestyle='--', label='Oversold 30')
    ax.axhline(50, color='black', linewidth=0.5, linestyle=':')
    ax.fill_between(dt, df['rsi_14'], 70, where=(df['rsi_14']>=70), alpha=0.2, color='red')
    ax.fill_between(dt, df['rsi_14'], 30, where=(df['rsi_14']<=30), alpha=0.2, color='green')
ax.set_ylabel('RSI(14)')
ax.legend(fontsize=7)
ax.set_ylim(0, 100)

# 3. MACD
ax = axes[2]
if 'macd_line' in df.columns:
    ax.plot(dt, df['macd_line'],   linewidth=1,   color='#2980b9', label='MACD')
    ax.plot(dt, df['macd_signal'], linewidth=0.9, color='#e74c3c', label='Signal')
    ax.bar(dt, df['macd_histogram'],
           color=['#2ecc71' if v >= 0 else '#e74c3c' for v in df['macd_histogram']],
           width=1, alpha=0.5, label='Histogram')
    ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('MACD')
ax.legend(fontsize=7)

# 4. Volume
ax = axes[3]
ax.bar(dt, df['Volume'], width=1, color='#3498db', alpha=0.6)
if 'volume_ma_30d' in df.columns:
    ax.plot(dt, df['volume_ma_30d'], color='orange', linewidth=1.2, label='30d MA')
ax.set_ylabel('Volume')
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation with target_7d (top predictors preview)
feat_target = btc_feat.dropna(subset=['target_7d'])
base = {'Date','Open','High','Low','Close','Volume','is_outlier','data_quality'}
feat_cols = [c for c in feat_target.columns
             if c not in base and not c.startswith('target') and not c.startswith('future')]

corr_with_target = feat_target[feat_cols + ['target_7d']].corr()['target_7d'].drop('target_7d')
corr_abs = corr_with_target.abs().sort_values(ascending=False)

top20 = corr_abs.head(20)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top20.index[::-1], corr_with_target[top20.index[::-1]],
               color=['#2ecc71' if v >= 0 else '#e74c3c'
                      for v in corr_with_target[top20.index[::-1]]])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with target_7d')
ax.set_title('BTC — Top 20 Features Correlated with 7-Day Target',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nTop 10 predictors:')
print(corr_abs.head(10).round(4).to_string())

## 14. Key Insights Summary

In [ ]:
# Print pre-generated insights document
insights_path = REPORTS_ROOT / 'Key_Insights.md'
with open(insights_path, encoding='utf-8') as f:
    print(f.read())

In [ ]:
# Cross-category summary statistics
summary = all_stats.groupby('category').agg(
    Assets=('sharpe_ratio', 'count'),
    Avg_Sharpe=('sharpe_ratio', 'mean'),
    Avg_Annual_Return=('annual_return_pct', 'mean'),
    Avg_Volatility=('ann_volatility', 'mean'),
    Avg_MaxDrawdown=('max_drawdown_pct', 'mean'),
).round(3)

display(summary)

In [ ]:
# Final data completeness summary
print('=== DATA COMPLETENESS SUMMARY ===')
total_cleaned  = sum(len(list((CLEANED_ROOT / c).glob('*.csv')))
                     for c in ['Crypto','Commodities','ETFs','Futures','Stocks'])
total_features = sum(len(list((FEATURES_ROOT / c).glob('*.csv')))
                     for c in ['Crypto','Commodities','ETFs','Futures','Stocks'])

print(f'  Cleaned CSVs    : {total_cleaned}')
print(f'  Feature CSVs    : {total_features}')
print(f'  EDA PDFs        : {len(list(REPORTS_ROOT.glob("EDA_*.pdf")))}')
print(f'  Quality reports : {len(list(REPORTS_ROOT.glob("Data_Quality_*.json")))}')
print()
print('Feature columns per asset: 64–78 (varies by context features available)')
print('Target variables: target_3d, target_5d, target_7d, target_10d, target_14d')
print('Ready for model training: Data/Features/**/*_features.csv')

In [ ]:
# Feature dictionary
feature_dict = [
    ('return_1d',           'Daily price return',               '(p_t - p_{t-1}) / p_{t-1}',    'All'),
    ('return_7d/30d',       '7 / 30-day price return',          '(p_t - p_{t-n}) / p_{t-n}',    'All'),
    ('log_return_1d',       'Log daily return',                  'ln(p_t / p_{t-1})',             'All'),
    ('momentum_7d/30d',     'Return percentile rank (90d win)', 'rolling rank pct',              'All'),
    ('sma_20/50/100/200',   'Simple Moving Averages',           'rolling mean(n)',               'All'),
    ('ema_12/26',           'Exponential Moving Averages',      'ewm(span=12/26)',               'All'),
    ('dist_from_sma_50',    'Distance from SMA50',              '(close - sma50) / sma50',       'All'),
    ('golden_cross',        'SMA50 > SMA200 flag',              'binary',                        'All'),
    ('rsi_14',              'Relative Strength Index',          'Wilder RSI(14)',                'All'),
    ('macd_line',           'MACD line',                        'ema12 - ema26',                 'All'),
    ('macd_histogram',      'MACD histogram',                   'macd_line - signal',            'All'),
    ('bb_upper/lower',      'Bollinger Bands',                  'SMA20 ± 2σ',                   'All'),
    ('bb_bandwidth',        'BB bandwidth',                     '(upper-lower)/middle',          'All'),
    ('bb_pct_b',            'BB %B position',                   '(close-lower)/(upper-lower)',   'All'),
    ('atr_14',              'Average True Range',               'ewm TR(14)',                    'All'),
    ('adx_14',              'Average Directional Index',        'Wilder ADX(14)',                'All'),
    ('stoch_k/d',           'Stochastic Oscillator',            '%K(14), %D(3)',                 'All'),
    ('obv',                 'On-Balance Volume',                'cumulative signed volume',      'All'),
    ('vwap_20d',            'VWAP 20-day',                      'sum(TP*V)/sum(V)',              'All'),
    ('vol_30d/60d',         'Annualised historical volatility', 'std(log_ret,n)*sqrt(252)',      'All'),
    ('parkinson_vol_30d',   'Parkinson HL estimator',           'HL log range',                  'All'),
    ('price_to_high_1yr',   'Price / 52-week high',             'close / rolling_high(252)',     'All'),
    ('day_range_pct',       'Intraday range %',                 '(high-low)/close',              'All'),
    ('gap_pct',             'Open gap vs prev close',           '(open - prev_close)/prev_close','All'),
    ('target_7d',           '7-day forward label',              '1 if p_{t+7} > p_t',           'All'),
    ('corr_btc_30d',        '30d rolling BTC correlation',      'rolling corr of returns',       'Crypto'),
    ('beta_btc_30d',        'Rolling beta vs BTC',              'cov(ret, mkt) / var(mkt)',      'Crypto'),
    ('corr_n50_30d',        '30d rolling Nifty50 correlation',  'rolling corr of returns',       'Stocks'),
]

dict_df = pd.DataFrame(feature_dict, columns=['Feature', 'Description', 'Formula', 'Asset Type'])
display(dict_df)